# Results for model: nvidia/llama3-chatqa-1.5-8b

In [ ]:
import polars as pl
import xgboost as xgb

# Load data from './jane_street_data/train.parquet'
df = pl.read_parquet('./jane_street_data/train.parquet')

# Calculate a global TOP_QUANTILE (top 15%) of 'feature_00' relative to'responder_6' across rolling batches of 'date_id'
df = df.with_column(
    (
        pl.col('feature_00').rank().over(
            pl.window.partition_by('responder_6').range_between(-np.inf, 0.15)
        ).cast(pl.Categorical)
    ).alias('top_quantile')
)

# Train an XGBoost Regressor on the target'responder_6'
dtrain = xgb.DMatrix(df.select('top_quantile','responder_6'))
model = xgb.train({'objective':'reg:linear'}, dtrain, 100)

# Save the model
model.save_model('xgb_model.json')